In [ ]:
import numpy as np
import pandas as pd
import re
from pandarallel import pandarallel
from pathlib import Path
import warnings
import yaml

pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## (1) matching_info

In [ ]:
qcc_matching = []
matching_info_path = Path(dataset_config['path_cnki'] + 'CNKI_standard_Qichacha/matching_info/')  # Convert string to Path

# Initialize a counter to track the number of files loaded
file_count = 0

# Sort the files in the directory by name before iterating
for p in sorted(matching_info_path.iterdir()):
    if p.suffix in ['.xls', '.xlsx']:  # Ensure only Excel files are loaded
        print('Loading', p)
        df_item = pd.read_excel(p, header=1)  # Read the .xls file
        qcc_matching.append(df_item)
        file_count += 1  # Increment the counter for each loaded file
    else:
        print(f'Skipping non-Excel file: {p}')

# Print the total number of files loaded
print(f'Total number of files loaded: {file_count}')

In [ ]:
qcc_matching_info = pd.concat(qcc_matching, ignore_index=True)  # Concatenate and reset the index
qcc_matching_info.rename(columns={'导入名称': 'firm_name', '匹配企业名称': 'firm_name_qcc'}, inplace=True)
qcc_matching_info

In [ ]:
qcc_matching_info = qcc_matching_info[qcc_matching_info['匹配结果'] != '失败匹配'].copy()
qcc_matching_info.drop(columns=['Unnamed: 4', '匹配结果', '失败原因'], inplace=True)
qcc_matching_info.drop_duplicates(inplace=True)
qcc_matching_info.head()

In [ ]:
qcc_matching_info.to_csv(dataset_config['path_cnki'] + 'CNKI_standard_Qichacha/matching_info.csv', index=False)

## (2) basic_info

In [ ]:
qcc_basic = []
basic_info_path = Path(dataset_config['path_cnki'] + 'CNKI_standard_Qichacha/basic_info/')  # Convert string to Path

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, message="Workbook contains no default style")

# Initialize a counter to track the number of files loaded
file_count = 0

# Sort the files in the directory by name before iterating
for p in sorted(basic_info_path.iterdir()):
    if p.suffix in ['.xls', '.xlsx']:  # Ensure only Excel files are loaded
        print('Loading', p)
        df_item = pd.read_excel(p, header=1)  # Read the .xls file
        qcc_basic.append(df_item)
        file_count += 1  # Increment the counter for each loaded file
    else:
        print(f'Skipping non-Excel file: {p}')

# Print the total number of files loaded
print(f'Total number of files loaded: {file_count}')

In [ ]:
qcc_basic_info = pd.concat(qcc_basic, ignore_index=True)  # Concatenate and reset the index
qcc_basic_info.rename(columns={'企业名称': 'firm_name_qcc'}, inplace=True)
qcc_basic_info['firm_name_qcc'] = qcc_basic_info['firm_name_qcc'].str.replace('\n', '', regex=False)
qcc_basic_info.drop_duplicates(inplace=True)
qcc_basic_info.head()

In [ ]:
qcc_basic_info.to_csv(dataset_config['path_cnki'] +'CNKI_standard_Qichacha/basic_info.csv', index=False)

## (3) corporate_info

In [ ]:
qcc_corporate = []
corporate_info_path = Path(dataset_config['path_cnki'] + 'CNKI_standard_Qichacha/corporate_info/')  # Convert string to Path

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, message="Workbook contains no default style")

# Initialize a counter to track the number of files loaded
file_count = 0

# Sort the files in the directory by name before iterating
for p in sorted(corporate_info_path.iterdir()):
    if p.suffix in ['.xls', '.xlsx']:  # Ensure only Excel files are loaded
        print('Loading', p)
        df_item = pd.read_excel(p, header=1)  # Read the .xls file
        qcc_corporate.append(df_item)
        file_count += 1  # Increment the counter for each loaded file
    else:
        print(f'Skipping non-Excel file: {p}')

# Print the total number of files loaded
print(f'Total number of files loaded: {file_count}')

In [ ]:
qcc_corporate_info = pd.concat(qcc_corporate, ignore_index=True)  # Concatenate and reset the index
qcc_corporate_info.drop(columns=['序号'], inplace=True)
qcc_corporate_info.rename(columns={'企业名称': 'firm_name_qcc'}, inplace=True)
qcc_corporate_info.dropna(subset=['firm_name_qcc'], inplace=True) # Resolve cases where one company maps to multiple parent groups.
qcc_corporate_info['firm_name_qcc'] = qcc_corporate_info['firm_name_qcc'].str.replace('\n', '', regex=False)
qcc_corporate_info.drop_duplicates(inplace=True)
qcc_corporate_info.head()

In [ ]:
qcc_corporate_info['所属集团'] = qcc_corporate_info['所属集团'].replace('-', np.nan)
qcc_corporate_info['所属集团'] = qcc_corporate_info['所属集团'].apply(lambda x: f"{x}-C" if pd.notna(x) else x)
qcc_corporate_info

In [ ]:
qcc_corporate_info.to_csv(dataset_config['path_cnki'] +'CNKI_standard_Qichacha/corporate_info.csv', index=False)